In [1]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.1/112.6 GB disk)


In [2]:
# Veri setini indireceğimiz klasörü oluşturdum
!mkdir datasets
%cd datasets

# Kaggle/Roboflow kaynaklı örnek bir araç segmentasyon veri setini indirip çıkarıyorum
!curl -L "https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128-seg.zip" > coco128-seg.zip
!unzip -q coco128-seg.zip
!rm coco128-seg.zip

%cd ..
print("Veri seti başarıyla indirildi ve hazır! ✅")

/content/datasets
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 6952k  100 6952k    0     0  8256k      0 --:--:-- --:--:-- --:--:-- 8256k
/content
Veri seti başarıyla indirildi ve hazır! ✅


In [4]:
!yolo task=segment mode=train data=coco128-seg.yaml model=yolov8n-seg.pt epochs=20 batch=16 imgsz=640

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco128-seg.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspe

In [5]:
import cv2
import numpy as np
from ultralytics import YOLO

# Eğittiğim en iyi modeli (ağırlıkları) yüklüyorum
model = YOLO('/content/runs/segment/train-2/weights/best.pt')

# Örnek bir trafik/araç videosu indiriyorum
!wget -O test_video.mp4 "https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4"

# Videoyu okumak ve kaydetmek için OpenCV ayarları
cap = cv2.VideoCapture("test_video.mp4")
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
out = cv2.VideoWriter('sonuc_videosu.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

# --- HOCANIN İSTEDİĞİ PARAMETRELER ---
HOCA_THRESHOLD = 0.30 # (0.2 ile 0.5 arası istenmişti)
KERNEL_BOYUTU = (5, 5) # Gürültüleri silmek için matris boyutu
kernel = np.ones(KERNEL_BOYUTU, np.uint8)

print("Video işleniyor, bu biraz zaman alabilir...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Modeli frame üzerinde çalıştırdım (Threshold parametresi burada conf= olarak veriliyor)
    results = model(frame, conf=HOCA_THRESHOLD, verbose=False)

    # Siyah bir arka plan oluşturdum (Lineer cebirde sıfırlar matrisi: np.zeros)
    mask_siyah_arkaplan = np.zeros_like(frame)

    if results[0].masks is not None:
        # Piksellerin maskelerini aldım
        masks = results[0].masks.data.cpu().numpy()

        # Tüm maskeleri birleştirip tek bir matris yapıyorum
        combined_mask = np.any(masks, axis=0).astype(np.uint8) * 255

        # Maskeyi video boyutuna getirdim (Matris boyutlandırma)
        combined_mask = cv2.resize(combined_mask, (width, height))

        # --- KERNEL İŞLEMİ (Morfolojik Açılma) ---
        # Ufak gürültüleri (yanlışlıkla beyaz olmuş 1-2 pikseli) silmek için kernel kullanıyorum
        temizlenmis_maske = cv2.morphologyEx(combined_mask, cv2.MORPH_OPEN, kernel)

        # Siyah arkaplan üzerine sadece tespit edilen nesneleri beyaz olarak çizdim
        mask_siyah_arkaplan[temizlenmis_maske == 255] = [255, 255, 255]

    # Frame'i videoya yazdım
    out.write(mask_siyah_arkaplan)

cap.release()
out.release()
print("İşlem tamam! 'sonuc_videosu.mp4' dosyasına sol taraftaki klasör simgesinden ulaşabilirsin. ✅")

--2026-05-03 11:05:04--  https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/car-detection.mp4 [following]
--2026-05-03 11:05:05--  https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/car-detection.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2811553 (2.7M) [application/octet-stream]
Saving to: ‘test_video.mp4’

test_video.mp4      100%[===================>]   2.68M  --.-KB/s    in 0.01s   

2026-05-03 11:05:05 (246 MB/s) - ‘test_video.mp4’ saved [281155